In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [ ]:
# อ่านไฟล์
df = pd.read_csv("../data/-63-66-.csv", encoding="utf-8-sig")

print("=== CHECK DTYPES ===")
print(df.dtypes)
print("====================")


df["ค่าข้อมูล"] = pd.to_numeric(
    df["ค่าข้อมูล"]
        .astype(str)
        .str.replace(",", "", regex=False),
    errors="coerce"
)

# ลบค่า NaN ถ้ามี
df = df.dropna(subset=["ค่าข้อมูล"])

# ลบค่า 0
df = df[df["ค่าข้อมูล"] != 0]


# แปลงปี
df["ปีคริสต์ศักราช"] = df["ปีงบประมาณ"] - 543
df["ปีคริสต์ศักราช"] = pd.to_datetime(df["ปีคริสต์ศักราช"], format="%Y")



# บันทึกไฟล์ clean

df.to_csv("../data/cleaned_data.csv", index=False, encoding="utf-8-sig")

print("✅ cleaned_data.csv เรียบร้อย")


# รวมยอดรายปี
df_total = df.groupby("ปีคริสต์ศักราช")["ค่าข้อมูล"].sum().reset_index()
df_total["ปี"] = df_total["ปีคริสต์ศักราช"].dt.year

print(df_total)



# สร้างโมเดล
X = df_total[["ปี"]]
y = df_total["ค่าข้อมูล"]

model = LinearRegression()
model.fit(X, y)



# Forecast
future_years_ad = [2025, 2026, 2027]
future_years_be = [year + 543 for year in future_years_ad]

y_pred = model.predict([[year] for year in future_years_ad])

forecast = pd.DataFrame({
    "ปีงบประมาณ": future_years_be,
    "prediction": y_pred
})

forecast.to_csv("../data/forecast.csv", index=False, encoding="utf-8-sig")

print("✅ forecast.csv เรียบร้อย")
forecast